# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All entities are referenced by their `@id` fields.

### Dataset Source
Croissant JSON-LD: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and prepare for record exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\nDataset: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
List all available record sets, their fields, and `@id`s.

In [ ]:
# Get all record sets
record_sets = dataset.metadata.record_set
if not record_sets:
    # fallback: try .recordSets (older mlcroissant) or direct property access
    try:
        record_sets = dataset.metadata.recordSets
    except AttributeError:
        record_sets = []

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    if 'field' in rs:
        print(f"  Fields:")
        for fld in rs['field']:
            if isinstance(fld, dict):
                print(f"    - {fld.get('name','N/A')} (@id: {fld['@id']})")
            else:
                print(f"    - @id: {fld}")
    print('')

### Example: Show some records from a main tabular RecordSet by its `@id`

In [ ]:
# Display a sample from the first RecordSet
if record_sets:
    main_recordset_id = record_sets[0]['@id']
    print(f"Sample records from RecordSet @id: {main_recordset_id}")
    for ix, record in enumerate(dataset.records(record_set=main_recordset_id)):
        print(record)
        if ix >= 2:
            break  # Limit the display

## 3. Data Extraction
Load data from each record set into pandas DataFrames using their `@id`.

In [ ]:
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rsid in record_set_ids:
    try:
        recs = list(dataset.records(record_set=rsid))
        if len(recs) > 0:
            df = pd.DataFrame(recs)
            dataframes[rsid] = df
            print(f"Loaded {len(df)} records from RecordSet '{rsid}' (columns: {list(df.columns)})")
        else:
            print(f"RecordSet '{rsid}' yielded no records.")
    except Exception as e:
        print(f"Failed loading records from '{rsid}': {e}")

if record_set_ids:
    first_id = record_set_ids[0]
    if first_id in dataframes:
        print(f"\nColumns in main RecordSet '{first_id}':\n{dataframes[first_id].columns.tolist()}")
        display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply basic processing using field `@id`s. Here, we'll identify a numeric field, filter records by a value, normalize, and group by another field using `@id` notation.

In [ ]:
# EDA: Find a numeric field for processing
import numpy as np
# We'll use the first DataFrame (main RecordSet) if present
if record_set_ids:
    main_df_id = record_set_ids[0]
    df = dataframes.get(main_df_id)
    if df is not None:
        numeric_candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
        if not numeric_candidates:
            # Try to coerce some columns to numeric in case they were loaded as strings
            for col in df.columns:
                try:
                    df[col] = pd.to_numeric(df[col], errors='ignore')
                except Exception:
                    pass
            numeric_candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
        print(f"Numeric fields: {numeric_candidates}")

        if numeric_candidates:
            numeric_field_id = numeric_candidates[0]  # Use the first numeric field
            threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 10
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records in '{numeric_field_id}' > {threshold}:")
            display(filtered_df[[numeric_field_id]].head())

            # Normalization
            filtered_df[numeric_field_id + '_normalized'] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / filtered_df[numeric_field_id].std()
            print(f"Normalized '{numeric_field_id}' for filtered records:")
            display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())
            
            # Pick a group/category field (prefer fields with <20 unique non-numeric values)
            group_candidate = None
            for col in df.columns:
                if col == numeric_field_id: continue
                if df[col].dtype == object:
                    nunique = df[col].nunique()
                    if 2 <= nunique <= 20:
                        group_candidate = col
                        break
            if group_candidate is not None:
                grouped_df = filtered_df.groupby(group_candidate).mean(numeric_only=True)
                print(f"Grouped data by '{group_candidate}':")
                display(grouped_df)
            else:
                print("No suitable group field found.")
        else:
            print("No numeric fields available for analysis.")
    else:
        print(f"No DataFrame loaded for RecordSet '{main_df_id}'.")

## 5. Visualization

Simple visualizations of field distributions. Here we plot a histogram of the first numeric field, and a boxplot grouping by the first found category field if both exist.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if record_set_ids:
    main_df_id = record_set_ids[0]
    df = dataframes.get(main_df_id)
    if df is not None and not df.empty:
        # Repeat candidate selection from above
        numeric_candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
        group_candidates = [col for col in df.columns if df[col].nunique() > 1 and df[col].nunique() < 20 and df[col].dtype == object]

        if numeric_candidates:
            field_id = numeric_candidates[0]
            plt.figure(figsize=(6,4))
            df[field_id].hist(bins=15)
            plt.title(f"Distribution of '{field_id}'")
            plt.xlabel(field_id)
            plt.ylabel("Frequency")
            plt.show()

            if group_candidates:
                group_id = group_candidates[0]
                plt.figure(figsize=(10,4))
                sns.boxplot(x=group_id, y=field_id, data=df)
                plt.title(f"'{field_id}' by '{group_id}'")
                plt.xlabel(group_id)
                plt.ylabel(field_id)
                plt.xticks(rotation=45)
                plt.show()

## 6. Conclusion

This notebook demonstrated loading and analyzing the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id` fields. Key steps included:
- Programmatically listing available record sets, fields, and their `@id`s
- Extracting records and basic EDA/normalization
- Grouping and visualization

Further analyses (e.g., statistical modeling or machine learning) can be built on top of these structured records using `mlcroissant` for seamless dataset access.